# Salesforce

A tool for interacting with Salesforce CRM from LangChain.

## Overview

The `langchain-salesforce` package connects LangChain to Salesforce CRM so you can
query data, manage records, and explore object schemas from your applications.

### Key Features

- **SOQL Queries**: Run SOQL queries, with `query_all` following pagination for large result sets
- **SOSL Search**: Search across objects with SOSL
- **Record Management**: Create, read, update, upsert, and delete records
- **Schema Exploration**: Describe object schemas, list available objects, and read field-level metadata
- **Async Support**: Every operation can be awaited
- **Environment Variable Support**: Credentials are loaded from the environment by default

## Setup

Install the package:

```bash
pip install langchain-salesforce
```

## Authentication

Set your Salesforce credentials as environment variables:

```bash
export SALESFORCE_USERNAME="your-username@company.com"
export SALESFORCE_PASSWORD="your-password"
export SALESFORCE_SECURITY_TOKEN="your-security-token"
export SALESFORCE_DOMAIN="login"  # Use "test" for sandbox environments
```

## Instantiation

In [ ]:
from langchain_salesforce import SalesforceTool

# Credentials are read from the environment. To pass them explicitly, use:
# SalesforceTool(username=..., password=..., security_token=..., domain=...)
tool = SalesforceTool()

## Invocation

Every call is a dict with an `operation` key plus the parameters that operation needs.

## Query

`query` returns the first batch of up to 2000 records. Use `query_all` when you need
every matching record, whatever the result size.

In [ ]:
query_result = tool.invoke(
    {"operation": "query", "query": "SELECT Id, Name, Email FROM Contact LIMIT 5"}
)

all_contacts = tool.invoke(
    {"operation": "query_all", "query": "SELECT Id FROM Contact"}
)

## Search

`search` runs a SOSL search across objects.

In [ ]:
search_result = tool.invoke(
    {
        "operation": "search",
        "search": "FIND {Acme} IN ALL FIELDS RETURNING Account(Id, Name)",
    }
)

## Describe an Object

Fetches the schema for a Salesforce object.

In [ ]:
describe_result = tool.invoke({"operation": "describe", "object_name": "Account"})

## Get Field Metadata

Retrieves metadata for a single field, including its type, label, picklist values,
and whether it is createable or updateable.

In [ ]:
field_metadata = tool.invoke(
    {"operation": "get_field_metadata", "object_name": "Contact", "field_name": "Email"}
)

## List Available Objects

Retrieves every object available in the Salesforce org.

In [ ]:
objects = tool.invoke({"operation": "list_objects"})

## Read a Record

Fetches a single record by ID, without writing SOQL.

In [ ]:
contact = tool.invoke(
    {"operation": "get", "object_name": "Contact", "record_id": "003XXXXXXXXXXXXXXX"}
)

## Create, Update, and Delete

Writes always return a dict. `create` returns Salesforce's `{"id": ..., "success": ...}`
payload, while `update`, `upsert`, and `delete` return
`{"id": ..., "success": True, "status_code": 204}`.

In [ ]:
created = tool.invoke(
    {
        "operation": "create",
        "object_name": "Contact",
        "record_data": {"LastName": "Doe", "Email": "doe@example.com"},
    }
)

updated = tool.invoke(
    {
        "operation": "update",
        "object_name": "Contact",
        "record_id": "003XXXXXXXXXXXXXXX",
        "record_data": {"Email": "updated@example.com"},
    }
)

deleted = tool.invoke(
    {"operation": "delete", "object_name": "Contact", "record_id": "003XXXXXXXXXXXXXXX"}
)

## Upsert

`upsert` creates or updates a record. Address it by Salesforce ID, or by an external
ID written as `ExternalIdField__c/value`. A `status_code` of `201` means the record
was created; `204` means it was updated.

In [ ]:
upserted = tool.invoke(
    {
        "operation": "upsert",
        "object_name": "Contact",
        "record_id": "External_Id__c/abc-123",
        "record_data": {"LastName": "Doe", "Email": "doe@example.com"},
    }
)

## Async

Every operation can be awaited with `ainvoke`.

In [ ]:
contacts = await tool.ainvoke(
    {"operation": "query", "query": "SELECT Id, Name FROM Contact LIMIT 5"}
)

## Chaining

In [ ]:
from langchain_anthropic import ChatAnthropic
from langchain_core.messages import HumanMessage

llm = ChatAnthropic(model="claude-sonnet-4-20250514")

contacts_result = tool.invoke(
    {
        "operation": "query",
        "query": "SELECT Id, Name, Email, Phone FROM Contact LIMIT 3",
    }
)

analysis_prompt = f"""
Please analyze the following Salesforce contact data and provide insights:

Contact Data: {contacts_result["records"]}

Please provide:
1. A summary of the contacts
2. Any patterns you notice
3. Suggestions for data quality improvements
"""

analysis_result = llm.invoke([HumanMessage(content=analysis_prompt)])

## Agent Use

The tool can be bound to a model and called directly by an agent.

In [ ]:
llm_with_tools = llm.bind_tools([tool])

response = llm_with_tools.invoke("How many contacts do we have in Salesforce?")

## API Reference

For comprehensive documentation and API reference, see:

- [langchain-salesforce README](https://github.com/colesmcintosh/langchain-salesforce/blob/main/README.md)
- [SalesforceTool API Documentation](https://python.langchain.com/docs/integrations/tools/salesforce/)
- [Simple Salesforce Documentation](https://simple-salesforce.readthedocs.io/en/latest/)

## Additional Resources

- [Salesforce SOQL Reference](https://developer.salesforce.com/docs/atlas.en-us.soql_sosl.meta/soql_sosl/)
- [Salesforce REST API Developer Guide](https://developer.salesforce.com/docs/atlas.en-us.api_rest.meta/api_rest/)
- [LangChain Tools Documentation](https://python.langchain.com/docs/modules/tools/)